In [11]:
import pandas as pd
import numpy as np
from pyarrow import dataset as ds
import pickle

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

### load data

In [12]:
dataset = ds.dataset("../data/renovation-data-all.parquet", format="parquet")
table = dataset.to_table(columns=None, use_threads=True)
df_preprocessed = table.to_pandas()
print(f"shape: {df_preprocessed.shape}")
df_preprocessed.head()

shape: (39736, 83)


,entity_id,epc_before,ontology.building.building_general.construction_year,ontology.building.building_general.floor_area,ontology.building.pv.available_roof_area_for_solar_panel_in_m2,ontology.building.pv.has_pv,steps,has_insulate_exterior_inclined_roof,has_insulate_exterior_flat_roof,has_insulate_exterior_wall,has_insulate_cavity_wall,has_insulate_floors_above_air_from_below,has_replace_single_and_old_double_for_hr_glass_with_new_frames,has_replace_non_condensing_gas_boiler_with_condensing_gas_boiler,has_replace_epump_with_dhw,has_install_pv_panels,steps_gt,step_count,ontology.building.energy_carriers.cooking.GAS_DISTRICT,ontology.building.energy_carriers.cooking.ELECTRICITY,ontology.building.energy_carriers.space_cooling.ELECTRICITY,ontology.building.systems.hot_water_system_types_on_electricity.ELECTRICAL_RESISTANCE,ontology.building.systems.hot_water_system_types_on_electricity.HEAT_PUMP,ontology.building.energy_carriers.space_heating.GAS_DISTRICT,ontology.building.energy_carriers.space_heating.COAL,ontology.building.energy_carriers.space_heating.OIL,ontology.building.energy_carriers.space_heating.ELECTRICITY,ontology.building.energy_carriers.space_heating.WOOD,ontology.building.energy_carriers.space_heating.PELLETS,ontology.building.envelope.floor_boundary_types.PLAIN_GROUND,ontology.building.end_uses.SPACECOOLING,ontology.building.end_uses.ELECTRICITY_PRODUCTION,ontology.building.energy_carriers.hot_water.COAL,ontology.building.energy_carriers.hot_water.GAS_DISTRICT,ontology.building.energy_carriers.hot_water.ELECTRICITY,ontology.building.energy_carriers.hot_water.OIL,ontology.building.energy_carriers.hot_water.PELLETS,ontology.building.energy_carriers.hot_water.WOOD,ontology.building.envelope.basement_occurance_ALL,ontology.building.envelope.basement_occurance_NONE,...,ontology.building.envelope.insulation_level_roof_extension_WELL_INSULATED,ontology.building.envelope.insulation_level_roof_main_NOT_APPLICABLE,ontology.building.envelope.insulation_level_roof_main_NOT_INSULATED,ontology.building.envelope.insulation_level_roof_main_UNKNOWN,ontology.building.envelope.insulation_level_roof_main_WELL_INSULATED,ontology.building.envelope.insulation_type_floor_above_basement_LIGHTLY_INSULATED,ontology.building.envelope.insulation_type_floor_above_basement_MODERATELY_INSULATED,ontology.building.envelope.insulation_type_floor_above_basement_NOT_INSULATED,ontology.building.envelope.insulation_type_walls_front_LIGHTLY_INSULATED,ontology.building.envelope.insulation_type_walls_front_MODERATELY_INSULATED,ontology.building.envelope.insulation_type_walls_front_NOT_INSULATED,ontology.building.envelope.insulation_type_walls_rear_LIGHTLY_INSULATED,ontology.building.envelope.insulation_type_walls_rear_MODERATELY_INSULATED,ontology.building.envelope.insulation_type_walls_rear_NOT_INSULATED,ontology.building.envelope.insulation_type_walls_sides_LIGHTLY_INSULATED,ontology.building.envelope.insulation_type_walls_sides_MODERATELY_INSULATED,ontology.building.envelope.insulation_type_walls_sides_NOT_APPLICABLE,ontology.building.envelope.insulation_type_walls_sides_NOT_INSULATED,ontology.building.building_general.building_type_SFH_DETACHED,ontology.building.building_general.building_type_SFH_TERRACED,ontology.building.systems.ventilation_type_MECHANICAL_DEMAND_CONTROLED,ontology.building.systems.ventilation_type_MECHANICAL_MINIMAL,ontology.building.systems.ventilation_type_NATURAL_WINDOWS_ONLY,ontology.building.systems.ventilation_type_NATURAL_WINDOW_VENTS,ontology.building.systems.space_heating_generation_systems_on_burning_fuel.CENTRAL_BOILER,ontology.building.envelope.window_glazing_type.DOUBLE_HIGH_EFFICIENCY_GLAZING,ontology.building.envelope.window_glazing_type.DOUBLE_OLD_GLAZING,ontology.building.envelope.window_glazing_type.SINGLE_GLAZING,ontology.building.envelope.window_glazing_type.TRIPLE_GLAZING,ontology.building.systems.space_heating_emission_systems.RADIATOR,ontology.building.systems.space_heating_emission_systems.CONVECTOR,ontology.

### train model

In [13]:
steps_encoding = {
    'insulate_exterior_inclined_roof': 1,
    'insulate_exterior_flat_roof': 2,
    'insulate_exterior_wall': 3,
    'insulate_cavity_wall': 4,
    'insulate_floors_above_air_from_below': 5,
    'replace_single_and_old_double_for_hr_glass_with_new_frames': 6,
    'replace_non_condensing_gas_boiler_with_condensing_gas_boiler': 7,
    'replace_epump_with_dhw': 8,
    'install_pv_panels': 9,  
}

def best_threshold_for_metric(y_true, y_proba, metric_fn):
    scores = {}
    thresholds = np.linspace(0, 1, 101)
    best_score = -1
    best_threshold = 0.5

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        s = metric_fn(y_true, y_pred)
        scores[t] = s
        if s > best_score:
            best_score = s
            best_threshold = t

    return best_threshold, best_score, scores

class ThresholdedModel:
    def __init__(self, model, threshold: float = 0.5, positive_class_index: int = 1):
        self.model = model
        self.threshold = float(threshold)
        self.positive_class_index = int(positive_class_index)

    def fit(self, X, y):
        # Delegate to underlying sklearn model
        return self.model.fit(X, y)
    
    def predict_proba(self, X):
        # Delegate to underlying sklearn model
        return self.model.predict_proba(X)

    def decision_function(self, X):
        # Probability for the positive class
        proba = self.predict_proba(X)
        return proba[:, self.positive_class_index]

    def predict(self, X):
        scores = self.decision_function(X)
        return (scores >= self.threshold).astype(int)

def get_columns_to_drop(df, component):
    columns_to_drop = ["entity_id"]

    if component == "relevant_steps":
        columns_to_drop.extend([col for col in df.columns if col.startswith("has_")])
        columns_to_drop.extend(["step_count", "steps", "steps_gt"])
    elif component == "first_step":
        columns_to_drop.extend(["first_step", "steps", "steps_gt"]) # remove steps(gt) since list, but same information is in 'has_' columns

    return columns_to_drop

In [14]:
models_RF = {}
reports = {}
oof_predictions = {}

RFC_params = {
    'random_state': 42,
}

# define features
columns_to_drop = get_columns_to_drop(df_preprocessed, component="relevant_steps")
X = df_preprocessed.drop(columns=columns_to_drop)
print(f"{X.shape[1]} features used for training: {X.columns.tolist()}")

for action in steps_encoding.keys():
    # define target
    target = f"has_{action}"
    y = df_preprocessed[target]

    ###################################
    ##### out-of-fold predictions #####
    ###################################

    # using StratifiedKFold to maintain class distribution in each fold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof_proba = np.zeros(len(y))

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_va = y[train_idx], y[valid_idx]

        RFC_model = RandomForestClassifier(**RFC_params)

        wrapped_model = ThresholdedModel(
            model=RFC_model,
            threshold=0.5,
            positive_class_index=1
        )

        wrapped_model.fit(X_tr, y_tr)

        # store OOF prediction p(class=1)
        oof_proba[valid_idx] = wrapped_model.predict_proba(X_va)[:, 1]

    # find best threshold for macro F1 score
    best_t, best_f1, scores = best_threshold_for_metric(y_true=y, y_proba=oof_proba, metric_fn=lambda yt, yp: f1_score(yt, yp, pos_label=1))

    # calculate F1 scores
    y_pred = (oof_proba >= best_t).astype(int)

    f1_pos = f1_score(y, y_pred, pos_label=1)
    f1_neg = f1_score(y, y_pred, pos_label=0)

    reports[target] = {
        "Threshold": float(best_t),
        "F1_positive": float(f1_pos),
        "F1_negative": float(f1_neg),
        "support_positive": int((y == 1).sum()),
        "support_negative": int((y == 0).sum())
    }

    # save oof predictions
    oof_predictions[target] = oof_proba

    ################################
    ##### training final model #####
    ################################
    RFC_model_final = RandomForestClassifier(**RFC_params)
    wrapped_model_final = ThresholdedModel(
        model=RFC_model_final,
        threshold=best_t,
        positive_class_index=1
    )
    wrapped_model_final.fit(X, y)
    models_RF[target] = wrapped_model_final

# save models to file
filename = '../models/relevant_steps_RF.pkl'
with open(filename, 'wb') as file:
    pickle.dump({
        "models_RF": models_RF,
        "oof_predictions": oof_predictions
    }, file)

print("Saved:", filename)

70 features used for training: ['epc_before', 'ontology.building.building_general.construction_year', 'ontology.building.building_general.floor_area', 'ontology.building.pv.available_roof_area_for_solar_panel_in_m2', 'ontology.building.pv.has_pv', 'ontology.building.energy_carriers.cooking.GAS_DISTRICT', 'ontology.building.energy_carriers.cooking.ELECTRICITY', 'ontology.building.energy_carriers.space_cooling.ELECTRICITY', 'ontology.building.systems.hot_water_system_types_on_electricity.ELECTRICAL_RESISTANCE', 'ontology.building.systems.hot_water_system_types_on_electricity.HEAT_PUMP', 'ontology.building.energy_carriers.space_heating.GAS_DISTRICT', 'ontology.building.energy_carriers.space_heating.COAL', 'ontology.building.energy_carriers.space_heating.OIL', 'ontology.building.energy_carriers.space_heating.ELECTRICITY', 'ontology.building.energy_carriers.space_heating.WOOD', 'ontology.building.energy_carriers.space_heating.PELLETS', 'ontology.building.envelope.floor_boundary_types.PLAIN_

### analyse results

In [15]:
results_df = pd.DataFrame(columns=["target", "F1 score (0)", "F1 score (1)", "Support (0)", "Support (1)"])
results_df["target"] = [key for key in reports.keys()]
results_df["F1 score (0)"] = [reports[key]["F1_negative"] for key in reports.keys()]
results_df["F1 score (1)"] = [reports[key]["F1_positive"] for key in reports.keys()]
results_df["Support (0)"] = [int(reports[key]["support_negative"]) for key in reports.keys()]
results_df["Support (1)"] = [int(reports[key]["support_positive"]) for key in reports.keys()]

display(results_df.round(3))

weighted_average = {
    '0': sum(reports[f"has_{key}"]["F1_negative"] * reports[f"has_{key}"]["support_negative"] for key in steps_encoding) / 
         sum(reports[f"has_{key}"]["support_negative"] for key in steps_encoding),
    '1': sum(reports[f"has_{key}"]["F1_positive"] * reports[f"has_{key}"]["support_positive"] for key in steps_encoding) / 
         sum(reports[f"has_{key}"]["support_positive"] for key in steps_encoding)
}

print(f"Weighted average F1 score (0): {weighted_average['0']:.3f}")
print(f"Weighted average F1 score (1): {weighted_average['1']:.3f}")
print(f"Mean weighted average F1 score: {np.mean([weighted_average['0'], weighted_average['1']]):.3f}")

,target,F1 score (0),F1 score (1),Support (0),Support (1)
0,has_insulate_exterior_inclined_roof,0.840,0.907,14967,24769
1,has_insulate_exterior_flat_roof,0.888,0.896,18273,21463
2,has_insulate_exterior_wall,0.853,0.907,15192,24544
3,has_insulate_cavity_wall,0.929,0.777,31366,8370
4,has_insulate_floors_above_air_from_below,0.677,0.922,8714,31022
5,has_replace_single_and_old_double_for_hr_glass...,0.891,0.909,18785,20951
6,has_replace_non_condensing_gas_boiler_with_con...,0.892,0.519,34310,5426
7,has_replace_epump_with_dhw,0.984,0.903,34058,5678
8,has_install_pv_panels,0.945,0.994,4071,35665


Weighted average F1 score (0): 0.899
Weighted average F1 score (1): 0.908
Mean weighted average F1 score: 0.903
